In [12]:
# requires:
# conda install -c conda-forge gdal
# conda install -c conda-forge libgdal-hdf4
# conda install rasterio                      <--- needs to be the conda version; pip version has only minimal GDAL and can't open HDF files

import os
from functools import lru_cache
import glob
import json
import numpy as np
import pandas as pd

import rasterio
from rasterio.warp import reproject, Resampling, transform_bounds
from rasterio.merge import merge
from rasterio.windows import from_bounds

import torch
from torch.utils.data import Dataset, DataLoader

In [13]:
from io import StringIO
import re

with open("sn_bound_10deg.txt") as f:
    raw = f.read()
lines = [ln for ln in raw.splitlines()
         if ln.strip() and re.match(r"^\d", ln.strip())]
buf = StringIO("\n".join(lines))
cols = ["iv", "ih", "lon_min", "lon_max", "lat_min", "lat_max"]
modis_grid = pd.read_csv(buf, delim_whitespace=True, header=None, names=cols)

def lat_lon_to_modis(ctr_lat, ctr_long):
    rows = modis_grid[ (modis_grid.lon_min <= ctr_long) & ( ctr_long <= modis_grid.lon_max) & (modis_grid.lat_min <= ctr_lat) & ( ctr_lat <= modis_grid.lat_max) ]
    return list(zip(rows.ih.values, rows.iv.values))

/tmp/ipykernel_2331064/851350310.py:10: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  modis_grid = pd.read_csv(buf, delim_whitespace=True, header=None, names=cols)


In [14]:
# TODO these should reallly be added as descriptions in the TIFFs

big_bands = [
    "temperature_2m",
    "dewpoint_temperature_2m",
    "total_precipitation",
    "surface_pressure",
    "u_component_of_wind_10m",
    "v_component_of_wind_10m",
    "surface_solar_radiation_downwards",
    "surface_thermal_radiation_downwards",
    "surface_net_solar_radiation",
    "surface_net_thermal_radiation",
    "surface_latent_heat_flux",
    "surface_sensible_heat_flux",
    "boundary_layer_height",
    "total_cloud_cover",
    "mean_sea_level_pressure",
]
med_bands = [
    "mod11a2_LST_Day_K", 
    "mod11a2_LST_Night_K",
    "mod13q1_NDVI", 
    "mod13q1_EVI",
    "mcd43a4_NDSI",
    "mcd43a4_BAI"
]


In [15]:
study = '5a280544-06db-4c85-9713-842547972edd' # '338388ea-6483-472c-9cdc-c2997e645df0'
df = pd.read_csv(glob.glob(f'mbr_studies_2year_filtered_traintest/{study}*')[0])

In [16]:
### rasterio methods ###

def _get_rasterio(path):
    with rasterio.open(path) as ds:
        arr = ds.read()
        return {
                    "array": arr,
                    "transform": ds.transform,
                    "crs": ds.crs,
                    "nodata": ds.nodata,
                    "bounds": ds.bounds,
                    "bands": ds.descriptions
                }

@lru_cache(maxsize=16)
def _get_rasterio_cached(path):
    return _get_rasterio(path)

@lru_cache(maxsize=20)
def _get_rasterio_window_cached(path, bounds):
    with rasterio.open(path) as ds:
        min_lon, min_lat, max_lon, max_lat = bounds
        window = from_bounds(min_lon, min_lat, max_lon, max_lat, ds.transform)
        return {
            "array": ds.read(window=window),
            "transform": ds.window_transform(window),
            "crs": ds.crs,
            "nodata": ds.nodata,
            "bounds": bounds, # rasterio.windows.bounds(window, ds.transform),
        }

@lru_cache(maxsize=20)
def _get_merged_rasterio_window_cached(paths, bounds_latlon):
    """Applies CRF bounds transform (to accomodate MODIS)
    """
    datasets = [rasterio.open(p) for p in paths]
    try:
        crs = datasets[0].crs
        descriptions = datasets[0].descriptions
        nodata_val = datasets[0].nodata or 255

        # transform to MODIS CRS
        left, bottom, right, top = transform_bounds(
            "EPSG:4326", # bounds assumed to come from EPSG:4326
            crs,         # but we want to use them in the target (e.g. MODIS) CRS
            *bounds_latlon,
        )

        mosaic, out_transform = merge(
            datasets,
            bounds=(left, bottom, right, top),
            nodata=nodata_val,
            resampling=rasterio.enums.Resampling.nearest,
        )
    except:
        print("Oh no! In _get_merged_rasterio_window_cached")
    finally:
        for ds in datasets:
            ds.close()

    return {
        "array": mosaic,           # (bands, H, W)
        "transform": out_transform,
        "crs": crs,
        "nodata": nodata_val,
        "bounds": (left, bottom, right, top),
        "bands": descriptions,
    }

def _reproject_all(layers):
    """Return all layers reprojected as float32."""
    ref_raster = layers[0]
    _, H_ref, W_ref = ref_raster['array'].shape
    
    aligned = [ ref_raster['array'] ]
    dst_crs = ref_raster['crs']
    dst_transform = ref_raster['transform']
    for layer in layers[1:]:
        # create new np array as destination - same channels as this layer but with target width/height
        dst = np.full((layer["array"].shape[0], H_ref, W_ref), np.nan, dtype=np.float32)
        reproject(
            source=layer["array"], # (C, H_src, W_src)
            destination=dst,
            src_transform=layer["transform"],
            src_crs=layer["crs"],
            dst_transform=dst_transform,
            dst_crs=dst_crs,
            resampling=Resampling.nearest,   # nearest or bilinear
        )
        aligned.append(dst)

    return aligned

In [116]:
class ChipDataset(Dataset):
    def __init__(self,  study_id, idx_to_timestamp, chip_base_dir='chips_out', dem_base_dir='dem_out'):
        self.idx_to_timestamp = idx_to_timestamp

        self.chip_dir = os.path.join(chip_base_dir, study_id)
        self.dem_dir  = os.path.join(dem_base_dir, study_id)
        self.dem_manifest = json.load(open(os.path.join(self.dem_dir, 'dem_manifest.json'), 'r'))

        self.hii_dir = '/archive/vision/beery/movebench/human_impact_index/'
        self.cvf_dir = '/archive/vision/beery/movebench/modis_cvf2015-2025/'
        self.lc_dir = '/archive/vision/beery/movebench/modis_lc2015-2025/'
        self.bioclim_dir = '/archive/vision/beery/movebench/bioclim/'

        # build modis filepath indices to avoid globs later
        # self.lc_dir + f"/MCD12Q1.A{year}001.h{h:02d}v{v:02d}*.hdf"
        self.modis_lc_index = {}
        for path in glob.glob(self.lc_dir + "/MCD12Q1.A*.h*v*.hdf"):
            fname = os.path.basename(path)
            year = int(fname.split(".A")[1][:4])
            h = int(fname.split(".h")[1][:2])
            v = int(fname.split("v")[1][:2])
            self.modis_lc_index[(int(year), int(h), int(v))] = path
        
        self.modis_cvf_index = {}
        for path in glob.glob(self.cvf_dir + "/MOD44B.A*.h*v*.hdf"):
            fname = os.path.basename(path)
            year = int(fname.split(".A")[1][:4])
            h = int(fname.split(".h")[1][:2])
            v = int(fname.split("v")[1][:2])
            self.modis_cvf_index[(int(year), int(h), int(v))] = path

    def __getitem__(self, idx):
        timestamp = self.idx_to_timestamp[idx]
        all_layers = []
        layer_names = []
        
        # ERA5 chips
        big_chip   = _get_rasterio(os.path.join(self.chip_dir, f"point_{idx}_big.tif"))
        med_chip   = _get_rasterio(os.path.join(self.chip_dir, f"point_{idx}_medium.tif"))
        # print("big:", big_chip['array'].shape)
        # print("med:", med_chip['array'].shape)
        # right now, small is just dynamic world, and it's broken
        # small_chip = _get_rasterio(os.path.join(self.chip_dir, f"point_{idx}_small.tif"))
        all_layers.extend([med_chip, big_chip])
        layer_names.extend(["era5_med", "era5_big"]) # TODO: would need med_bands, big_bands to access these later for e.g. derived features

        # get location for other queries -- assume small will be the most accurate
        bounds = med_chip['bounds']
        center_lon = (bounds[0] + bounds[2]) / 2
        center_lat = (bounds[1] + bounds[3]) / 2

        # DEM - bigger tile
        for chunk in self.dem_manifest.get("chunk_info", []):
            b = chunk["bounds"]
            if b["min_lon"] <= center_lon <= b["max_lon"] and b["min_lat"] <= center_lat <= b["max_lat"]:
                demf = os.path.join(self.dem_dir, chunk["filename"])
                dem_raster = _get_rasterio_cached(demf)
                # dem_raster = _get_rasterio_window_cached(demf, med_chip['bounds'])
                break
        # print("DEM", dem_raster['array'].shape)
        all_layers.append(dem_raster)
        layer_names.append("DEM")
        
        # MODIS land cover -- annual, hdf files
        # int-valued (should be one-hot as features)
        year = int(max(min(timestamp.year, 2025), 2015))
        # get all files that the bounds fall into - will need to merge them
        hvs = []
        for lat in [ bounds[1], bounds[3] ]:
            for lon in [ bounds[0], bounds[2] ]:
                hvs.extend(lat_lon_to_modis(lat, lon))
        sd_paths = []
        for (h, v) in hvs:
            hdf_path_i = self.modis_lc_index[(year, int(h), int(v))]
            sd_paths.append("HDF4_EOS:EOS_GRID:" + hdf_path_i + ":MCD12Q1:LC_Type1")
        lc_raster = _get_merged_rasterio_window_cached(tuple(sd_paths), bounds)
        # print("MODIS LC", lc_raster['array'].shape)
        all_layers.append(lc_raster)
        layer_names.append("modis_lc")
        
        # MODIS vegetation cover -- annual
        cvf_rasters = []
        for band in [':MOD44B_250m_GRID:Percent_Tree_Cover', 
                     ':MOD44B_250m_GRID:Percent_NonTree_Vegetation',
                     ':MOD44B_250m_GRID:Percent_NonVegetated']:
            sd_paths = []
            for (h, v) in hvs:
                hdf_path_i = self.modis_cvf_index[(year, int(h), int(v))]
                sd_paths.append("HDF4_EOS:EOS_GRID:" + hdf_path_i + band)
            # print("merging", sd_paths)
            cvf_raster = _get_merged_rasterio_window_cached(tuple(sd_paths), bounds)
            cvf_rasters.append(cvf_raster)
        # print("MODIS CVF", [cvf_raster['array'].shape for cvf_raster in cvf_rasters])
        all_layers.extend(cvf_rasters)
        layer_names.extend(["modis_treecover", "modis_nontreeveg", "modis_nonveg"])

        # Human impact -- annual
        year = max(min(timestamp.year, 2020), 2015)
        hii_f = os.path.join(self.hii_dir, f'hii_{year}-01-01.tif')
        hii_raster = _get_rasterio_window_cached(hii_f, med_chip['bounds'])
        # print("HII", hii_raster['array'].shape)
        all_layers.append(hii_raster)
        layer_names.append("HII")

        # BioClim -- no time; whole world
        bioclim_fs = [ os.path.join(self.bioclim_dir, f"wc2.1_30s_bio_{i}.tif") for i in range(1,20) ] 
        bioclim_rasters = [ _get_rasterio_window_cached(bioclim_f, med_chip['bounds']) for bioclim_f in bioclim_fs ]
        # print("bioclim", [raster['array'].shape for raster in bioclim_rasters])
        all_layers.extend(bioclim_rasters)
        layer_names.extend([ f"bioclim_{i}" for i in range(1,20) ])

        # reproject all to same resolution/CRS/bounds
        # and convert to torch
        reprojected = _reproject_all(all_layers)
        reprojected_t =[
            torch.from_numpy(l) for l in reprojected
        ]

        # TODO add derived types - e.g. uplift
        # ...later...

        # convert classification data to one-hot
        lc_idx = layer_names.index("modis_lc")
        reprojected_t[lc_idx] = torch.nn.functional.one_hot(reprojected_t[lc_idx].long() - 1, # one-indexed -> zero-indexed 
                                                            num_classes=17).squeeze(0).permute(2,0,1)
        nodata_vals  = [ a['nodata'] or torch.inf for a in all_layers ] # need to 'or' to catch None values, which mess up the equality test
        valid_masks = [ (layer != nodata_val) for layer, nodata_val in zip(reprojected_t, nodata_vals) ]
        
        return torch.cat(reprojected_t), torch.cat(valid_masks)

    def __len__(self):
        return len(self.idx_to_timestamp)

In [117]:
# import time
ds = ChipDataset(study, idx_to_timestamp=pd.to_datetime(df.timestamp))
ds[100]

(tensor([[[ 2.7218e+02,  2.7218e+02,  2.7218e+02,  ...,        -inf,
                  -inf,        -inf],
          [ 2.7218e+02,  2.7218e+02,  2.7218e+02,  ...,        -inf,
                  -inf,        -inf],
          [ 2.7210e+02,  2.7210e+02,  2.7218e+02,  ...,        -inf,
                  -inf,        -inf],
          ...,
          [ 2.7312e+02,  2.7308e+02,  2.7308e+02,  ...,        -inf,
                  -inf,        -inf],
          [ 2.7284e+02,  2.7284e+02,  2.7284e+02,  ...,        -inf,
                  -inf,        -inf],
          [ 2.7284e+02,  2.7284e+02,  2.7284e+02,  ...,        -inf,
                  -inf,        -inf]],
 
         [[ 2.6484e+02,  2.6484e+02,  2.6484e+02,  ...,        -inf,
                  -inf,        -inf],
          [ 2.6484e+02,  2.6484e+02,  2.6484e+02,  ...,        -inf,
                  -inf,        -inf],
          [ 2.6480e+02,  2.6480e+02,  2.6484e+02,  ...,        -inf,
                  -inf,        -inf],
          ...,
    

In [ ]:
from tqdm import tqdm

dataloader = DataLoader(ds, 
                        batch_size=1,
                        shuffle=False, 
                        num_workers=16)

# get values for standardization
layer_sum = None
layer_sq_sum = None
count = None
for i_batch, (x, valid_mask) in tqdm(enumerate(dataloader), total=len(ds)):
    x = x.squeeze(0)
    valid_mask = valid_mask.squeeze(0)
    num_valid  = valid_mask.sum(dim=(-1, -2))  # (C,)
    x_clean = torch.where(valid_mask, x, torch.zeros_like(x))

    # Per-layer sums
    s  = x_clean.sum(dim=(-1, -2))         # (C,)
    s2 = (x_clean ** 2).sum(dim=(-1, -2))  # (C,)

    if layer_sum is None:
        layer_sum = s
        layer_sq_sum = s2
        count = num_valid
    else:
        layer_sum += s
        layer_sq_sum += s2
        count += num_valid

    # TESTING
    if i_batch > 20:
        break

print("Done")

# Compute mean and variance per layer over all samples
mean = layer_sum / count
var  = layer_sq_sum / count - mean**2

# Clamp tiny negative variances due to float rounding
# var = torch.clamp(var, min=0.0)
std = torch.sqrt(var)

mean, std

/data/vision/beery/scratch/justin/envs/movement/lib/python3.11/site-packages/torch/utils/data/dataloader.py:626: UserWarning: This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 8, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
  1%|          | 19/3687 [00:10<21:22,  2.86it/s] 